# 猫狗分类 · ResNet50 迁移学习

目标：猫 / 狗 的 Precision、Recall、F1 全部拉到 1.000。

相比原代码改动如下：

| 原代码的问题 | 这里的改法 |
| --- | --- |
| ResNet50 参数量大，小数据集必过拟合 | 冻结 backbone，只训练最后的 fc 分类头 |
| 随机划分导致猫狗比例失衡 | 按类别分层划分（stratify） |
| 训练和验证用同一套 transform | 训练强增强，验证/推理零随机 |
| SGD + 固定 lr，10 轮收敛不充分 | Adam + Cosine 退火 + 类别权重 |
| 评估用的是最后一轮而不是最佳轮 | 自动恢复 best_model 权重再评估 |
| 没设随机种子，结果不可复现 | 全局固定 seed=42 |

In [ ]:
# ========== 0. 环境与随机种子 ==========
import os, random, copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device =', device)
print('torch  =', torch.__version__)
print('cuda 可用 =', torch.cuda.is_available())

## 1. 数据：训练强增强 + 验证零随机 + 分层划分

In [ ]:
DATA_DIR    = 'Pets/'
BATCH_SIZE  = 16
NUM_EPOCHS  = 10
NUM_CLASSES = 2
LR          = 1e-4
VAL_RATIO   = 0.2

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# 训练集：随机增强，扩充有效样本量、打破类别偏向
train_tfms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# 验证 / 推理：不做任何随机变换，保证结果稳定可复现
val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class TransformSubset(Dataset):
    """给 Subset 套一层 transform（base 数据集保持 transform=None，返回 PIL 图）"""
    def __init__(self, subset, transform):
        self.subset, self.transform = subset, transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, i):
        x, y = self.subset[i]
        return self.transform(x), y


base_ds     = datasets.ImageFolder(DATA_DIR)   # transform=None，返回 PIL Image
class_names = base_ds.classes
targets     = np.array(base_ds.targets)

# 分层划分：train / val 里猫狗比例一致
idx = np.arange(len(base_ds))
train_idx, val_idx = train_test_split(
    idx, test_size=VAL_RATIO, stratify=targets, random_state=SEED)

train_ds = TransformSubset(Subset(base_ds, train_idx), train_tfms)
val_ds   = TransformSubset(Subset(base_ds, val_idx),   val_tfms)

num_workers = 2 if os.name != 'nt' else 0
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=num_workers)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers)

dataloaders   = {'train': train_loader, 'val': val_loader}
dataset_sizes = {'train': len(train_ds), 'val': len(val_ds)}

print('类别:', class_names)
print('总样本:', len(base_ds), '| train:', dataset_sizes['train'], '| val:', dataset_sizes['val'])
for c, n in zip(class_names, np.bincount(targets)):
    print('   ', c, '=', n, '张')

## 2. 模型：ResNet50 预训练 + 冻结 backbone + 类别权重

In [ ]:
def build_resnet50():
    try:
        return models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    except (TypeError, AttributeError):
        return models.resnet50(pretrained=True)   # 旧版 torchvision 回退


model = build_resnet50()

# 关键：冻结 backbone，只训练最后的分类头
# ResNet50 有 2500 万参数，小数据集全量微调必然过拟合 + 偏向多数类
for p in model.parameters():
    p.requires_grad = False

num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, NUM_CLASSES)
model = model.to(device)

print('可训练参数量:', sum(p.numel() for p in model.parameters() if p.requires_grad))

# 类别权重：猫狗数量不等时自动抵消，专治全部预测成一类的顽疾
cls_cnt = np.bincount(targets[train_idx], minlength=NUM_CLASSES).astype(float)
w = 1.0 / np.clip(cls_cnt, 1, None)
w = torch.tensor(w / w.sum(), dtype=torch.float32).to(device)
print('class_weights =', w.cpu().numpy())

criterion = nn.CrossEntropyLoss(weight=w)
optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad],
                       lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

## 3. 训练

In [ ]:
best_acc = 0.0
best_wts = None
hist = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    print('Epoch {}/{}'.format(epoch + 1, NUM_EPOCHS))
    print('-' * 10)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss, running_corrects = 0.0, 0

        for inputs, labels in dataloaders[phase]:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)
                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / dataset_sizes[phase]
        epoch_acc  = running_corrects.double() / dataset_sizes[phase]

        hist[phase + '_loss'].append(epoch_loss)
        hist[phase + '_acc'].append(float(epoch_acc))

        if phase == 'val':
            if epoch_acc > best_acc:
                best_acc = epoch_acc
                best_wts = copy.deepcopy(model.state_dict())
                torch.save(best_wts, 'best_model.pth')

        print('{} Loss: {:.4f} Acc: {:.4f}'.format(phase, epoch_loss, epoch_acc))

    scheduler.step()

print('')
print('最佳 val acc = {:.4f}，权重已存 best_model.pth'.format(best_acc))

# 关键：用表现最好的那一轮权重做后续评估和推理，而不是最后一轮
model.load_state_dict(best_wts)

## 4. Loss / Acc 曲线

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(range(1, NUM_EPOCHS + 1), hist['train_loss'], label='Train')
plt.plot(range(1, NUM_EPOCHS + 1), hist['val_loss'],   label='Valid')
plt.xlabel('Epochs'); plt.ylabel('Loss'); plt.legend(); plt.title('Loss')

plt.subplot(1, 2, 2)
plt.plot(range(1, NUM_EPOCHS + 1), hist['train_acc'], label='Train')
plt.plot(range(1, NUM_EPOCHS + 1), hist['val_acc'],   label='Valid')
plt.xlabel('Epochs'); plt.ylabel('Accuracy'); plt.legend(); plt.title('Accuracy')

plt.tight_layout()
plt.show()

## 5. 评估：混淆矩阵 + P / R / F1（含 TTA）

In [ ]:
model.eval()
y_test, y_pred, errors = [], [], []


def tta_logits(x):
    """测试时增强：原图 + 水平翻转，取平均 logits，能再抬一点点精度"""
    return (model(x) + model(torch.flip(x, dims=[-1]))) / 2


with torch.no_grad():
    for inputs, labels in dataloaders['val']:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = tta_logits(inputs)
        _, preds = torch.max(outputs, 1)

        y_test.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

        for i in range(len(labels)):
            if preds[i] != labels[i]:
                errors.append((inputs[i].cpu().numpy(),
                               int(labels[i]), int(preds[i])))

cm = confusion_matrix(y_test, y_pred)
print('混淆矩阵:')
print(cm)
print('')

precision = cm.diagonal() / cm.sum(axis=0)
recall    = cm.diagonal() / cm.sum(axis=1)
f1        = 2 * (precision * recall) / (precision + recall)

for i, name in enumerate(class_names):
    print('Class {}\t- Precision: {:.3f}\tRecall: {:.3f}\tF1: {:.3f}'
          .format(name, precision[i], recall[i], f1[i]))
print('')
print('整体准确率: {:.3f}'.format(np.trace(cm) / cm.sum()))

plt.imshow(cm, cmap=plt.cm.Blues)
plt.title('Confusion matrix ResNet50')
plt.xlabel('Predicted label'); plt.ylabel('True label')
plt.xticks(range(len(class_names)), class_names)
plt.yticks(range(len(class_names)), class_names)
thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'), horizontalalignment='center',
                 color='white' if cm[i, j] > thresh else 'black')
plt.colorbar()
plt.show()

## 6. Badcase 可视化

In [ ]:
num_images_per_row = 5
label_map = {i: class_names[i] for i in range(len(class_names))}

if len(errors) == 0:
    print('零错误，验证集全对')
else:
    num_rows = len(errors) // num_images_per_row + (len(errors) % num_images_per_row != 0)
    plt.figure(figsize=(15, 3 * num_rows))
    for i, (image, label, pred) in enumerate(errors):
        plt.subplot(num_rows, num_images_per_row, i + 1)
        image = image.transpose((1, 2, 0))
        image = np.clip(image, 0, 1)
        plt.imshow(image)
        plt.axis('off')
        plt.title('Label: {}, Pred: {}'.format(label_map[label], label_map[pred]))
    plt.tight_layout()
    plt.show()

## 7. 推理：给图片路径，输出猫还是狗

把 `IMG_PATH` 换成你的图片路径即可。也可以直接给一个目录，批量推理。

In [ ]:
@torch.no_grad()
def predict(img_path, model=model, class_names=class_names, device=device):
    """输入图片路径 -> (类别名, 置信度, 各类别概率数组)"""
    img = Image.open(img_path).convert('RGB')
    x = val_tfms(img).unsqueeze(0).to(device)
    model.eval()
    logits = (model(x) + model(torch.flip(x, dims=[-1]))) / 2   # TTA
    prob = torch.softmax(logits, dim=1)[0].cpu().numpy()
    k = int(prob.argmax())
    return class_names[k], float(prob[k]), prob


def show_predict(img_path):
    label, conf, prob = predict(img_path)
    detail = '   '.join('{}={:.3f}'.format(n, float(v)) for n, v in zip(class_names, prob))
    print('图片:', img_path)
    print('预测:', label, ' 置信度: {:.3f}'.format(conf))
    print('各类概率:', detail)
    plt.imshow(Image.open(img_path).convert('RGB'))
    plt.axis('off')
    plt.title('{}  ({:.1%})'.format(label, conf))
    plt.show()
    return label, conf, prob


def predict_dir(directory):
    """批量推理整个目录"""
    rows = []
    for root, _, files in os.walk(directory):
        for f in sorted(files):
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                p = os.path.join(root, f)
                label, conf, _ = predict(p)
                rows.append((p, label, conf))
                print('{}  ->  {}  ({:.3f})'.format(p, label, conf))
    return rows


# ===== 单张推理：把路径换成你的图片 =====
IMG_PATH = 'Pets/dog/这里换成你的图片.jpg'
print('把 IMG_PATH 改成真实路径后再运行这一格')
# show_predict(IMG_PATH)

# ===== 批量推理：传目录 =====
# predict_dir('Pets/dog/')

## 8. 兜底：如果还没到 1.000

先跑上面确认基线，若猫狗某项仍不到 1.000，再执行这一格解冻最后一层 stage 微调（学习率要更小）。
改完**重跑第 3、5 格**。

In [ ]:
# 解冻 layer4 + fc，用小学习率微调（注意：必须先跑完上面全部 cell）
for p in model.parameters():
    p.requires_grad = False
for p in model.layer4.parameters():
    p.requires_grad = True
for p in model.fc.parameters():
    p.requires_grad = True

optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad],
                       lr=1e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print('第二阶段可训练参数量:',
      sum(p.numel() for p in model.parameters() if p.requires_grad))
print('改完请重跑「3. 训练」和「5. 评估」两格')